# Mesin Pencari — BM25 + Stopword + Stemming + Query Expansion

Peningkatan pada sistem dasar `07_mesin_pencari.ipynb`: **ekspansi sinonim** dari
`Data/kamus_sinonim.csv` (10 pasangan). Term query yang punya padanan di kamus
ditambah dengan sinonimnya berbobot **0,5** sebelum dicari dengan BM25.

- `use_sinonim=False` → Versi A (baseline).
- `use_sinonim=True` → Versi B (baseline + sinonim).

In [1]:
import re
import sys
import pandas as pd

sys.path.insert(1, "notebooks")  # bila notebook dijalankan dari root proyek
pd.set_option("display.max_colwidth", 120)

from stki_utils import (load_corpus, corpus_texts, preprocess, BM25Model,
                        load_synonyms, expand_query,
                        load_queries, load_qrels, evaluate)


class MesinPencari:
    def __init__(self, use_sinonim=True, use_stopword=True, use_stemming=True,
                 w_sinonim=0.5):
        self.korpus = load_corpus().set_index("doc_id")
        self.opsi = dict(use_stopword=use_stopword, use_stemming=use_stemming)
        docs = {d: preprocess(t, **self.opsi) for d, t in corpus_texts().items()}
        self.model = BM25Model(docs)
        self.use_sinonim = use_sinonim
        self.w_sinonim = w_sinonim
        self.sinonim = load_synonyms(use_stemming=use_stemming) if use_sinonim else {}

    def bobot_query(self, query):
        """Token query -> {term: bobot}; sinonim ditambahkan dengan bobot w_sinonim."""
        q = preprocess(query, **self.opsi)
        if not self.use_sinonim:
            return {t: 1.0 for t in q}
        return expand_query(q, self.sinonim, weight=self.w_sinonim)

    def cari(self, query, k=10):
        return [d for d, _ in self.model.search_weighted(self.bobot_query(query), k=k)]

    def tampilkan(self, query, k=3):
        q_terms = set(preprocess(query, **self.opsi))
        print(f"Hasil pencarian untuk: {query!r}\n")
        for rank, (d, s) in enumerate(self.model.search_weighted(self.bobot_query(query), k=k), 1):
            row = self.korpus.loc[d]
            kalimat = re.split(r"(?<=\.)\s+", row.isi)
            # cuplikan: kalimat dengan term query terbanyak
            cuplikan = max(kalimat, key=lambda s_: len(q_terms & set(preprocess(s_, **self.opsi))))
            print(f"{rank}. [{d}] {row.judul}  (skor {s:.3f})\n   ...{cuplikan}\n")

## Demo: Q03

Query Q03 (*strategi transformasi digital UMKM Indonesia*) mendapat tambahan
`digitalisasi` dari kamus. Bandingkan top-5 Versi A dan Versi B.

In [2]:
mesin_a = MesinPencari(use_sinonim=False)
mesin_b = MesinPencari(use_sinonim=True)

query = "strategi transformasi digital UMKM Indonesia"
tambahan = {t: w for t, w in mesin_b.bobot_query(query).items()
            if t not in preprocess(query)}

print("Term tambahan dari kamus:", tambahan)
print("Peringkat Versi A:", mesin_a.cari(query, k=5))
print("Peringkat Versi B:", mesin_b.cari(query, k=5), "\n")
mesin_b.tampilkan(query, k=5)

Term tambahan dari kamus: {'digitalisasi': 0.5}
Peringkat Versi A: ['K008', 'K009', 'K048', 'K004', 'K003']
Peringkat Versi B: ['K008', 'K009', 'K048', 'K006', 'K010'] 

Hasil pencarian untuk: 'strategi transformasi digital UMKM Indonesia'

1. [K008] Segera Atasi Kendala dalam Transformasi Digital di Sektor UMKM  (skor 7.372)
   ...Artikel MPR menyoroti 70,2 persen UMKM menghadapi kendala saat bertransformasi menggunakan teknologi digital.

2. [K009] Satu Dekade, Transformasi Digital UMKM Dorong Pertumbuhan Ekonomi Nasional  (skor 7.254)
   ...Komdigi mencatat satu dekade transformasi digital telah mendorong pertumbuhan ekonomi nasional melalui UMKM.

3. [K048] Bappenas Dorong Inovasi Ekonomi Keuangan untuk Transformasi Digital  (skor 6.000)
   ...Bappenas mendorong inovasi ekonomi keuangan untuk mendukung transformasi digital.

4. [K006] Pemerintah Terus Genjot Digitalisasi UMKM untuk Percepat Pemulihan Ekonomi Nasional  (skor 5.450)
   ...Sekitar 8 juta UMKM sudah terdigitalisasi pad

## Evaluasi 25 query (k = 5)

In [3]:
queries, qrels = load_queries(), load_qrels()
baris = []
for nama, opsi in [("BM25 dasar", dict(use_sinonim=False)),
                   ("+ sinonim", dict(use_sinonim=True))]:
    m = MesinPencari(**opsi)
    run = {q: m.cari(t) for q, t in queries.items()}
    rata = evaluate(run, qrels, k=5).loc["Rata-rata"]
    baris.append({"konfigurasi": nama, "P@5": rata["P@5"], "MAP": rata["AP"], "nDCG@5": rata["nDCG@5"]})
pd.DataFrame(baris)

,konfigurasi,P@5,MAP,nDCG@5
0,BM25 dasar,0.944,0.9103,0.9370
1,+ sinonim,0.968,0.9295,0.9613


**Hasil.** Ekspansi sinonim menaikkan P@5 0,944 → **0,968**, MAP 0,9103 → **0,9295**,
dan nDCG@5 0,9370 → **0,9613**. Peningkatan terbesar pada Q20 (`konektivitas → jaringan, infrastruktur`,
AP +0,288) dan Q03 (`transformasi → digitalisasi`, AP +0,114). Dua query turun karena sinonim terlalu
umum dan menarik dokumen lintas subtopik: Q25 (AP −0,021) dan Q07 (`literasi → kecakapan`, AP −0,013).
Rincian ada di `docs/15_hasil_eksperimen.md`.